In [2]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import accuracy_score

In [2]:
if not all(name in globals() for name in ("df_train", "df_val", "TARGET")):
    print("Data setup is missing. Load df_train, df_val, and TARGET before running diagnostics.")
else:
    df_train[TARGET] = df_train[TARGET].astype("string")
    df_val[TARGET] = df_val[TARGET].astype("string")

    print("Train path:", TRAIN_PATH)
    print("Validation path:", VAL_PATH)
    print("Target:", TARGET)
    print("Train shape / non-null labels:", df_train.shape, df_train[TARGET].notna().sum())
    print("Validation shape / non-null labels:", df_val.shape, df_val[TARGET].notna().sum())
    print("Train ID dtype / sample:", df_train[TARGET].dtype, df_train[TARGET].dropna().head(5).tolist())
    print("Validation ID dtype / sample:", df_val[TARGET].dtype, df_val[TARGET].dropna().head(5).tolist())

Data setup is missing. Load df_train, df_val, and TARGET before running diagnostics.


In [3]:
required_setup = ("train_sample", "val_sample", "TARGET", "all_features", "cat_cols")
if not all(name in globals() for name in required_setup):
    print("Model setup is missing. Define train_sample, val_sample, TARGET, all_features, and cat_cols before training.")
else:
    train_sample[TARGET] = train_sample[TARGET].astype("string")
    val_sample[TARGET] = val_sample[TARGET].astype("string")

    train_labeled = train_sample.dropna(subset=[TARGET]).copy()
    val_labeled = val_sample.dropna(subset=[TARGET]).copy()

    if train_labeled.empty:
        print("Skipping CatBoost: the older training dataset has no labeled AnimalId values.")
    else:
        train_pool = Pool(train_labeled[all_features], train_labeled[TARGET], cat_features=cat_cols or None)
        val_pool = None
        if not val_labeled.empty:
            val_pool = Pool(val_labeled[all_features], val_labeled[TARGET], cat_features=cat_cols or None)

        model = CatBoostClassifier(iterations=300, verbose=100, random_seed=42)
        model.fit(train_pool, eval_set=val_pool)

        if not val_labeled.empty:
            preds = model.predict(val_pool).ravel()
            print(f"Validation Accuracy: {accuracy_score(val_labeled[TARGET], preds) * 100:.2f}%")
        else:
            print("No labeled validation rows are available for accuracy scoring.")

Model setup is missing. Define train_sample, val_sample, TARGET, all_features, and cat_cols before training.
